# FinanceGPT - Dataset Preparation Notebook

This notebook is used for:
- Loading finance-related datasets
- Cleaning and filtering data
- Formatting prompts
- Tokenizing text
- Preparing datasets for model training

## Installing Required Libraries

In [1]:
!pip install datasets transformers -q

## Importing Required Libraries

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

## Loading Finance Dataset

In [3]:
dataset = load_dataset("gbharti/finance-alpaca")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 68912
    })
})


## Viewing First Dataset Sample

In [4]:
print(dataset["train"][0])

{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The short

## Viewing Multiple Dataset Samples

In [5]:
for i in range(3):

    print(dataset["train"][i])

    print("-" * 80)

{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The short

## Understanding Dataset Fields

In [6]:
sample = dataset["train"][0]

print("INSTRUCTION:")
print(sample["instruction"])

print("\nOUTPUT:")
print(sample["output"][:300])

INSTRUCTION:
For a car, what scams can be plotted with 0% financing vs rebate?

OUTPUT:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car


## Prompt Formatting

The dataset must be converted into prompt-response format
so the model can learn conversational behavior.

In [7]:
def format_prompt(example):

    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

In [8]:
formatted_dataset = dataset["train"].map(
    format_prompt
)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/68912 [00:00<?, ? examples/s]

### Instruction:
For a car, what scams can be plotted with 0% financing vs rebate?

### Response:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The shorter length m

## Filtering Long Responses

Very large outputs are removed to keep training efficient.

In [9]:
def is_good_example(example):

    return len(example["output"]) < 500

In [10]:
filtered_dataset = dataset["train"].filter(
    is_good_example
)

print("Original dataset size:", len(dataset["train"]))

print("Filtered dataset size:", len(filtered_dataset))

Filter:   0%|          | 0/68912 [00:00<?, ? examples/s]

Original dataset size: 68912
Filtered dataset size: 47685


## Formatting Filtered Dataset

In [11]:
formatted_dataset = filtered_dataset.map(
    format_prompt
)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/47685 [00:00<?, ? examples/s]

### Instruction:
Why does it matter if a Central Bank has a negative rather than 0% interest rate?

### Response:
That is kind of the point, one of the hopes is that it incentivizes banks to stop storing money and start injecting it into the economy themselves. Compared to the European Central Bank investing directly into the economy the way the US central bank has been doing. (The Federal Reserve buying mortgage backed securities) On a country level, individual European countries have tried this before in recent times with no noticeable effect.


## Defining Finance Keywords

In [12]:
finance_keywords = [
    "stock",
    "market",
    "finance",
    "investment",
    "bank",
    "revenue",
    "profit",
    "earnings",
    "ratio",
    "debt",
    "equity",
    "shares",
    "trading",
    "economy",
    "company"
]

## Filtering Finance Related Samples

In [13]:
def is_finance_related(example):

    text = (
        example["instruction"].lower()
        +
        " "
        +
        example["output"].lower()
    )

    return any(
        keyword in text
        for keyword in finance_keywords
    )

In [14]:
finance_dataset = filtered_dataset.filter(
    is_finance_related
)

print(
    "Filtered finance dataset size:",
    len(finance_dataset)
)

Filter:   0%|          | 0/47685 [00:00<?, ? examples/s]

Filtered finance dataset size: 6794


## Tokenization

Tokenization converts text into numerical tokens
that AI models can understand.

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "gpt2"
)

tokenizer.pad_token = tokenizer.eos_token

## Tokenizing Prompt Dataset

In [16]:
def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [19]:
tokenized_dataset = formatted_dataset.map(
    tokenize_function
)

Map:   0%|          | 0/47685 [00:00<?, ? examples/s]

## Inspecting Tokenized Dataset

In [20]:
print(tokenized_dataset[0].keys())

dict_keys(['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'])


In [21]:
print(tokenized_dataset[0]["input_ids"][:20])

[21017, 46486, 25, 198, 5195, 857, 340, 2300, 611, 257, 5694, 5018, 468, 257, 4633, 2138, 621, 657, 4, 1393]


## Understanding Tokenization Output

input_ids:
- numerical representation of text tokens

attention_mask:
- tells the model which tokens are important
- ignores padding tokens

## Splitting Dataset

The dataset is divided into:
- training dataset
- testing dataset

This helps evaluate model performance later.

In [22]:
split_dataset = tokenized_dataset.train_test_split(
    test_size=0.1
)

In [23]:
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 42916
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 4769
    })
})


## Saving Processed Dataset

In [24]:
tokenized_dataset.save_to_disk(
    "financegpt_dataset"
)

Saving the dataset (0/1 shards):   0%|          | 0/47685 [00:00<?, ? examples/s]

## Final Observation

The dataset has now been:
- loaded
- inspected
- filtered
- formatted
- tokenized
- split for training

This processed dataset is now ready for future fine-tuning.

FinanceGPT now has a proper training data pipeline.